# 00 — Pré-processamento

Entrada: `data/raw/Coffee_Data_Set.csv` e `data/raw/crop_yield.csv`.
Saída: `data/processed/clima_limpo.csv`, `perfil_regiao.csv` e `crop_yield_limpo.csv`.

Etapas: carga → tipos → nulos → duplicados → inconsistências → outliers → features → salvar.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

coffee = pd.read_csv('../data/raw/Coffee_Data_Set.csv', parse_dates=['Date'])
crop = pd.read_csv('../data/raw/crop_yield.csv')
print('clima:', coffee.shape, '| rendimento:', crop.shape)
coffee.dtypes

clima: (233000, 19) | rendimento: (1000000, 10)


LON                        float64
LAT                        float64
YEAR                         int64
MM                           int64
DD                           int64
DOY                          int64
Date                datetime64[us]
Temp_Max                   float64
Temp_Min                   float64
Humidity                   float64
Solar_Radiation            float64
Precipitation_mm           float64
Location                       str
Close_USD_60kg             float64
Is_Weekend                    bool
Is_Interpolated               bool
brl                        float64
cny                        float64
mxn                        float64
dtype: object

## 1. Seleção de colunas
Só entram as colunas que respondem às perguntas do projeto. Ficam de fora o preço (`Close_USD_60kg`, `brl`, `cny`, `mxn`: igual para todas as regiões na mesma data, não mede produtividade), as colunas de calendário redundantes com `Date` e as flags `Is_Weekend`/`Is_Interpolated`. No rendimento, só `Rainfall_mm`, `Temperature_Celsius` e o alvo, pois são as únicas variáveis que existem também no dataset de clima.

In [2]:
coffee = coffee[['Location', 'LAT', 'LON', 'Date', 'Temp_Max', 'Temp_Min', 'Humidity', 'Solar_Radiation', 'Precipitation_mm']]
crop = crop[['Rainfall_mm', 'Temperature_Celsius', 'Yield_tons_per_hectare']]

## 2. Valores nulos

In [3]:
print(coffee.isna().sum()[lambda s: s > 0])
print(crop.isna().sum()[lambda s: s > 0])

Solar_Radiation    28
dtype: int64
Series([], dtype: int64)


`Solar_Radiation` tem poucos nulos (28 em 233 mil). Como é uma série diária por região, interpolamos no tempo dentro de cada região, sem misturar regiões.

In [4]:
coffee = coffee.sort_values(['Location', 'Date'])
coffee['Solar_Radiation'] = coffee.groupby('Location')['Solar_Radiation'].transform(lambda s: s.interpolate().bfill().ffill())
print('nulos restantes:', coffee.isna().sum().sum())

nulos restantes: 0


## 3. Duplicados
No clima, a chave natural é (região, data).

In [5]:
print('clima:', coffee.duplicated(['Location', 'Date']).sum(), '| rendimento:', crop.duplicated().sum())
coffee = coffee.drop_duplicates(['Location', 'Date'])
# no rendimento, linhas iguais são amostras diferentes que coincidem por acaso (valores contínuos); só removemos se houver
crop = crop.drop_duplicates()

clima: 0 | rendimento: 0


## 4. Inconsistências
Regras de domínio: temperatura mínima não pode passar da máxima, chuva e rendimento não podem ser negativos, umidade fica entre 0 e 100.

In [6]:
regras = {
    'Temp_Min > Temp_Max': (coffee['Temp_Min'] > coffee['Temp_Max']).sum(),
    'chuva < 0': (coffee['Precipitation_mm'] < 0).sum(),
    'umidade fora de 0-100': (~coffee['Humidity'].between(0, 100)).sum(),
    'rendimento < 0': (crop['Yield_tons_per_hectare'] < 0).sum(),
}
print(regras)

coffee = coffee[(coffee['Temp_Min'] <= coffee['Temp_Max']) & (coffee['Precipitation_mm'] >= 0) & coffee['Humidity'].between(0, 100)]
crop = crop[crop['Yield_tons_per_hectare'] >= 0]
print(coffee.shape, crop.shape)

{'Temp_Min > Temp_Max': np.int64(0), 'chuva < 0': np.int64(0), 'umidade fora de 0-100': np.int64(0), 'rendimento < 0': np.int64(231)}
(233000, 9) (999769, 3)


## 5. Outliers
Contagem pelo IQR e boxplot. **Não removemos**: chuva e temperatura extremas são fenômenos reais do clima e removê-los apagaria exatamente os eventos que interessam. Chuva é naturalmente assimétrica (muitos dias secos, poucos dias de chuva forte).

In [7]:
cols = ['Temp_Max', 'Temp_Min', 'Humidity', 'Solar_Radiation', 'Precipitation_mm']
for c in cols:
    q1, q3 = coffee[c].quantile([.25, .75])
    i = q3 - q1
    n = ((coffee[c] < q1 - 1.5 * i) | (coffee[c] > q3 + 1.5 * i)).sum()
    print(f'{c}: {n} ({n / len(coffee):.1%})')

coffee[cols].plot(kind='box', subplots=True, layout=(1, 5), figsize=(14, 4))
plt.tight_layout()
plt.savefig('../data/processed/boxplot_clima.png')
plt.show()

Temp_Max: 1476 (0.6%)
Temp_Min: 2881 (1.2%)
Humidity: 5239 (2.2%)
Solar_Radiation: 1160 (0.5%)
Precipitation_mm: 19906 (8.5%)


C:\Users\ninja\AppData\Local\Temp\ipykernel_23028\1157611391.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Novas variáveis
- `pais` e `regiao`: separados de `Location` (formato `Pais_Regiao`).
- `temperatura_media` = (máxima + mínima) / 2.
- `perfil_regiao`: média por região, com `precipitacao_anual` (média diária × 365,25). Isso é necessário porque o `crop_yield.csv` usa chuva em mm de ciclo/ano (100–1000), não chuva diária.

In [8]:
partes = coffee['Location'].str.split('_')
coffee['pais'] = partes.str[0]
coffee['regiao'] = partes.str[1:].str.join(' ')
coffee['temperatura_media'] = (coffee['Temp_Max'] + coffee['Temp_Min']) / 2

clima = coffee.rename(columns={'LAT': 'latitude', 'LON': 'longitude', 'Date': 'data', 'Precipitation_mm': 'precipitacao', 'Humidity': 'umidade', 'Solar_Radiation': 'radiacao_solar'})
clima = clima[['pais', 'regiao', 'latitude', 'longitude', 'data', 'temperatura_media', 'precipitacao', 'umidade', 'radiacao_solar']]

perfil = clima.groupby(['pais', 'regiao'], as_index=False).agg(
    latitude=('latitude', 'first'), longitude=('longitude', 'first'),
    temperatura_media=('temperatura_media', 'mean'), precipitacao=('precipitacao', 'mean'),
    umidade=('umidade', 'mean'), radiacao_solar=('radiacao_solar', 'mean'))
perfil['precipitacao_anual'] = perfil['precipitacao'] * 365.25
perfil

,pais,regiao,latitude,longitude,temperatura_media,precipitacao,umidade,radiacao_solar,precipitacao_anual
0,Brazil,Espirito Santo Afonso,-20.0421,-40.9743,22.758541,2.979250,77.425737,17.566441,1088.171063
1,Brazil,Espirito Santo Conceicao,-20.3633,-41.2502,22.169075,3.148545,78.749172,17.464283,1150.006085
2,Brazil,Minas Gerais Aqua Boa,-17.9910,-42.3906,23.152938,2.711495,71.376330,18.069560,990.373416
3,Brazil,Minas Gerais Manhuacu,-20.2579,-46.1628,21.729783,3.357127,71.864791,18.666263,1226.190494
4,Brazil,Minas Gerais Patrocinio,-18.9349,-46.9923,21.873539,3.332626,71.223442,19.327799,1217.241477
5,Brazil,Minas Gerais Varginha,-21.5500,-45.4300,21.286519,3.370547,73.900008,18.426632,1231.092369
6,Brazil,Sao Paulo Campinas,-22.7884,-47.0579,21.338881,3.437258,75.749688,18.524389,1255.458306
7,Brazil,Sao Paulo Sao Jose,-20.5333,-47.5833,22.886925,3.665230,69.512648,19.072793,1338.725116
8,China,Puer,22.6555,100.9535,19.591259,4.286512,80.563735,16.496470,1565.648436
9,Colombia,Andes,5.6959,-75.8469,18.354262,6.317454,87.449386,17.183164,2307.450023


## 7. Salvar
A padronização (`StandardScaler`) fica dentro do K-Means (03) e do pipeline da regressão (04), ajustada só no treino, para não vazar informação do teste.

In [9]:
clima.to_csv('../data/processed/clima_limpo.csv', index=False)
perfil.to_csv('../data/processed/perfil_regiao.csv', index=False)
crop.to_csv('../data/processed/crop_yield_limpo.csv', index=False)
print(clima.shape, perfil.shape, crop.shape)

(233000, 9) (25, 9) (999769, 3)
